# Depuración y partición propia de PlantDoc — sesiones 5 y 5d (candidato, sin modelos)

Reglas del equipo, aplicadas **antes** del split sobre el pool de las 12 clases: (a) grupos de duplicados (phash ≤ 6) con etiquetas contradictorias → se excluyen **todas** las copias; (b) grupos de una sola clase → se conserva una copia (la primera por ruta ordenada); (c) pares de distancia 7–10 → los revisa **un** revisor (D3, 2026-10-09): `misma_foto` se une a los grupos de duplicados (valen (a) y (b)), `distinta` no se toca, vacío → PROVISIONAL; (d) revisión humana de `Tomato leaf` y `Bell_pepper leaf` con **un** revisor (D1, 2026-10-09): se excluye una imagen si `revisor_1` = `descartar` (`revisor_2` se ignora), fila vacía → PROVISIONAL. Luego dev/test 70/30 estratificado por clase, semilla 42.
El resultado es **COMPLETO** solo si la planilla y los pares 7–10 están completos; si no, PROVISIONAL (y `congelar_splits.py` no congela).
Nada se escribe en `data/splits/`. Config: `configs/01b_depuracion_plantdoc.yaml`.

In [ ]:
import json, subprocess, time
import numpy as np, pandas as pd, yaml
import matplotlib; matplotlib.use("Agg")
from foliares.utils.paths import cargar_rutas
from foliares.data import plantdoc_chequeos as pc, plantdoc_depurado as dp
from foliares.data.taxonomia import CLASES_EVAL_PD
try:
    import psutil; _proc = psutil.Process()
except ImportError:
    _proc = None
def mem_mb(): return round(_proc.memory_info().rss / 2**20) if _proc else None
pd.set_option("display.width", 250, "display.max_rows", 200, "display.max_columns", 40, "display.max_colwidth", 70)
R = cargar_rutas()
CFG = yaml.safe_load((R.repo / "configs" / "01b_depuracion_plantdoc.yaml").read_text(encoding="utf-8"))
h = subprocess.check_output(["git", "-C", str(R.plantdoc), "log", "-1", "--format=%H"], text=True).strip()
print("PlantDoc", h, "OK" if h == CFG["commit_plantdoc"] else "NO COINCIDE")
FIG = R.repo / CFG["salida"]["figuras"]; FIG.mkdir(parents=True, exist_ok=True)
T0 = time.time()
df, info = dp.construir_depurado(R, R.repo / CFG["renombrados_csv"], R.interim / CFG["cache_hashes"], R.repo / CFG["planilla_revision"],
                                 CFG["semilla"], CFG["frac_test"], CFG["umbral_duplicado"],
                                 pares_csv=R.repo / CFG["salida"]["pares_7_10"], revision=CFG["revision_etiquetas"], pares=CFG["pares_7_10"])
P, P6 = info["P"], info["P6"]
print(len(df), "imágenes en las 13 carpetas;", len(CLASES_EVAL_PD), "clases en el pool")

## Revisión humana de etiquetas (d)

In [ ]:
rev, par = info["rev"], info["par"]
print("Modo de revisión:", info["modo_revision"], "| pendiente:", info["pendiente"] or "nada")
print("Planilla (D1, un revisor):", rev.estado)
print("Imágenes excluidas por revisión (`revisor_1` = descartar):", len(rev.excluir))
for aviso in rev.avisos:
    print("AVISO:", aviso)
inv = rev.invalidas
if len(inv):
    print("¡Valores fuera del vocabulario (no se interpretan, esas filas NO excluyen)!"); display(inv)
display(pd.Series(rev.conteo["revisor_1"], name="revisor_1").to_frame())
print("Pares 7-10 (D3):", par.estado, "| decisiones:", par.conteo or "(sin listado)")
print("Pares `misma_foto` unidos a los grupos de duplicados:", len(par.misma_foto))

## 2a. Pares contradictorios por combinación de clases (pares phash ≤ 6 entre clases distintas, pool de 12 clases)

In [3]:
M = dp.matriz_contradictorios(df, P6)
print("Por combinación de clases del GRUPO (grupos e imágenes; `en_test_oficial` = copias en la carpeta test oficial):")
display(M["por_combinacion"])
print("Totales: grupos", int(M["por_combinacion"].grupos.sum()), "| imágenes", int(M["por_combinacion"].imagenes.sum()))
print("\nPor par de clases (número de PARES <=6):"); display(M["por_clase_par"])
print("\nMatriz de pares (simétrica):"); display(M["matriz_pares"])
# ¿Algún par con Tomato leaf o Bell_pepper leaf?
dud = ["Tomato leaf", "Bell_pepper leaf"]
con_dudosas = M["pares"][M["pares"].clase_par.map(lambda c: bool(set(c.split(" | ")) & set(dud)))] if len(M["pares"]) else M["pares"]
print("Pares contradictorios que involucran Tomato leaf o Bell_pepper leaf:", len(con_dudosas))
print("Imágenes de esas dos clases dentro de CUALQUIER grupo de duplicados:", int(((df.clase.isin(dud)) & (df.tam_grupo_hash > 1)).sum()))

Por combinación de clases del GRUPO (grupos e imágenes; `en_test_oficial` = copias en la carpeta test oficial):


,combinacion,grupos,imagenes,en_test_oficial
0,Potato leaf early blight | Potato leaf late blight,8,16,3
1,Potato leaf early blight | Tomato Early blight leaf,4,9,1
2,Potato leaf early blight | Tomato leaf late blight,3,6,0
3,Tomato Septoria leaf spot | Tomato leaf bacterial spot,3,6,1
4,Potato leaf late blight | Tomato leaf late blight,2,4,0
5,Tomato Early blight leaf | Tomato Septoria leaf spot,2,4,0
6,Potato leaf early blight | Potato leaf late blight | Tomato leaf l...,1,3,1
7,Bell_pepper leaf spot | Tomato Septoria leaf spot,1,2,0
8,Potato leaf early blight | Tomato Septoria leaf spot,1,2,0
9,Potato leaf late blight | Tomato Early blight leaf,1,2,0


Totales: grupos 29 | imágenes 60

Por par de clases (número de PARES <=6):


,clase_par,pares
0,Potato leaf early blight | Potato leaf late blight,9
1,Potato leaf early blight | Tomato Early blight leaf,5
2,Potato leaf early blight | Tomato leaf late blight,4
3,Potato leaf late blight | Tomato leaf late blight,3
4,Tomato Septoria leaf spot | Tomato leaf bacterial spot,3
5,Tomato Early blight leaf | Tomato Septoria leaf spot,2
6,Potato leaf early blight | Tomato Septoria leaf spot,1
7,Bell_pepper leaf spot | Tomato Septoria leaf spot,1
8,Potato leaf late blight | Tomato Early blight leaf,1
9,Tomato Early blight leaf | Tomato leaf late blight,1



Matriz de pares (simétrica):


,Bell_pepper leaf,Bell_pepper leaf spot,Potato leaf early blight,Potato leaf late blight,Tomato Early blight leaf,Tomato Septoria leaf spot,Tomato leaf,Tomato leaf bacterial spot,Tomato leaf late blight,Tomato leaf mosaic virus,Tomato leaf yellow virus,Tomato mold leaf
Bell_pepper leaf,0,0,0,0,0,0,0,0,0,0,0,0
Bell_pepper leaf spot,0,0,0,0,0,1,0,0,0,0,0,0
Potato leaf early blight,0,0,0,9,5,1,0,0,4,0,0,0
Potato leaf late blight,0,0,9,0,1,0,0,0,3,0,0,0
Tomato Early blight leaf,0,0,5,1,0,2,0,0,1,0,0,0
Tomato Septoria leaf spot,0,1,1,0,2,0,0,3,0,0,0,0
Tomato leaf,0,0,0,0,0,0,0,0,0,0,0,0
Tomato leaf bacterial spot,0,0,0,0,0,3,0,0,0,0,0,0
Tomato leaf late blight,0,0,4,3,1,0,0,0,0,0,0,0
Tomato leaf mosaic virus,0,0,0,0,0,0,0,0,0,0,1,0


Pares contradictorios que involucran Tomato leaf o Bell_pepper leaf: 0
Imágenes de esas dos clases dentro de CUALQUIER grupo de duplicados: 0


In [4]:
# Un ejemplo visual por combinación frecuente (>= N pares): el par de menor distancia (y menor ruta)
cl_de = dict(zip(df.ruta, df.clase))
x = P6[(P6.categoria == pc.CATEGORIAS[3])].copy()
x = x[x.ruta_a.isin(cl_de) & x.ruta_b.isin(cl_de)]
x = x[x.ruta_a.map(lambda r: cl_de.get(r) in CLASES_EVAL_PD) & x.ruta_b.map(lambda r: cl_de.get(r) in CLASES_EVAL_PD)]
x["clase_par"] = [" | ".join(sorted((a, b))) for a, b in zip(x.clase_a, x.clase_b)]
frec = M["por_clase_par"][M["por_clase_par"].pares >= CFG["frecuente_min_pares"]].clase_par.tolist()
ej = pd.concat([x[x.clase_par == c].sort_values(["distancia", "ruta_a"]).head(1) for c in frec]).reset_index(drop=True)
pc.grilla_pares(ej, R, FIG / "ejemplos_contradictorios.png", f"Un ejemplo por combinación frecuente (>= {CFG['frecuente_min_pares']} pares): {len(ej)} combinaciones", pares_por_fila=3)
display(ej[["clase_par", "distancia", "split_a", "nombre_original_a", "split_b", "nombre_original_b"]])
test_vistas = set(ej.ruta_a[ej.split_a == "test"]) | set(ej.ruta_b[ej.split_b == "test"])
print("Figura:", (FIG / "ejemplos_contradictorios.png").relative_to(R.repo).as_posix(), "| imágenes de test oficial en la figura:", len(test_vistas))

,clase_par,distancia,split_a,nombre_original_a,split_b,nombre_original_b
0,Potato leaf early blight | Potato leaf late blight,0,train,1421_0.jpeg?itok=FMtmgePj.jpg,test,1421_0.jpeg?itok=FMtmgePj.jpg
1,Potato leaf early blight | Tomato Early blight leaf,0,train,EBTomato.jpg,train,36-early-blight-concentric-rings.jpg
2,Potato leaf early blight | Tomato leaf late blight,0,train,LBleaf2INIA_0.jpg,train,Blght.jpg
3,Potato leaf late blight | Tomato leaf late blight,0,train,IMG_5813.jpg,train,IMG_5813-1024x768.jpg
4,Tomato Septoria leaf spot | Tomato leaf bacterial spot,0,train,Sept9-300x225.jpg,train,FS547-4-big.jpg
5,Tomato Early blight leaf | Tomato Septoria leaf spot,0,train,MPC105550.jpg.rendition.largest.ss.jpg,train,MPC105550.jpg.rendition.largest.550.jpg


Figura: docs/figuras_sesion5/ejemplos_contradictorios.png | imágenes de test oficial en la figura: 1


## 2b. Pares de rango 7–10 (los revisa un revisor; `misma_foto` se une a los grupos, el resto no se toca)

In [ ]:
p710 = P[(P.distancia >= CFG["umbral_duplicado"] + 1)]
p710 = p710[p710.ruta_a.map(lambda r: cl_de.get(r) in CLASES_EVAL_PD) & p710.ruta_b.map(lambda r: cl_de.get(r) in CLASES_EVAL_PD)].reset_index(drop=True)
pc.grilla_pares(p710, R, FIG / "pares_7_10.png", f"Pares de distancia 7-10 ({len(p710)}) — para revisión del equipo", pares_por_fila=3)
cols = ["categoria", "distancia", "split_a", "clase_a", "ruta_a", "nombre_original_a", "split_b", "clase_b", "ruta_b", "nombre_original_b"]
out = p710[cols].assign(decision_equipo="")
estado_listado = dp.guardar_pares_7_10(out, R.repo / CFG["salida"]["pares_7_10"])   # nunca pisa el archivo ni su `decision_equipo`
print("Listado de pares 7-10:", estado_listado)
display(p710[["categoria", "distancia", "clase_a", "nombre_original_a", "clase_b", "nombre_original_b"]])
print(len(p710), "pares; ninguno de test oficial:", not ((p710.split_a == "test") | (p710.split_b == "test")).any(),
      "| listado:", CFG["salida"]["pares_7_10"], "| figura:", (FIG / "pares_7_10.png").relative_to(R.repo).as_posix())

## 2c. Manifiesto depurado y partición propuesta

In [ ]:
cols = ["ruta", "clase", "split_oficial", "nombre_original", "grupo_hash", "tam_grupo_hash", "clases_en_grupo",
        "copia_conservada", "estado", "particion_propuesta"]
df[cols].assign(revision_estado=info["estado_revision"]).to_csv(R.interim / CFG["salida"]["manifiesto"], index=False, lineterminator="\n")
meta = dp.meta_depurado(df, info, CFG["semilla"], CFG["frac_test"], CFG["umbral_duplicado"], CFG["commit_plantdoc"], R.repo,
                        "notebooks/01b_depuracion_plantdoc.ipynb")
(R.interim / CFG["salida"]["meta"]).write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8", newline="\n")
print("Guardado:", R.relativa(R.interim / CFG["salida"]["manifiesto"]), "| meta:", R.relativa(R.interim / CFG["salida"]["meta"]))
display(df.estado.value_counts().rename("imágenes").to_frame())
print("Por carpeta oficial y estado:"); display(df.groupby(["split_oficial", "estado"]).size().unstack(fill_value=0))

In [7]:
t = dp.tabla_por_clase(df, CLASES_EVAL_PD); display(t)
A = CFG["alertas"]; al = dp.alertas(t, A["min_test_por_clase"], A["min_dev_por_clase"])
print(f"Alertas (test < {A['min_test_por_clase']} o dev < {A['min_dev_por_clase']}):", "NINGUNA" if al.empty else "")
if len(al): display(al)
print("Mínimos: dev =", int(t[t.clase != 'TOTAL'].dev.min()), "| test =", int(t[t.clase != 'TOTAL'].test.min()))

,clase,pool_antes,pool_despues,excl_contradictoria,excl_duplicada,excl_revision,dev,test
0,Tomato leaf bacterial spot,110,103,3,4,0,72,31
1,Tomato Early blight leaf,88,76,9,3,0,53,23
2,Tomato leaf late blight,111,99,7,5,0,69,30
3,Tomato mold leaf,91,90,1,0,0,63,27
4,Tomato Septoria leaf spot,151,138,7,6,0,97,41
5,Tomato leaf yellow virus,76,72,2,2,0,50,22
6,Tomato leaf mosaic virus,54,53,1,0,0,37,16
7,Tomato leaf,63,63,0,0,0,44,19
8,Potato leaf early blight,117,98,17,2,0,69,29
9,Potato leaf late blight,105,90,12,3,0,63,27


Alertas (test < 10 o dev < 30): NINGUNA
Mínimos: dev = 37 | test = 16


In [8]:
# Procedencia del test propuesto: ¿qué parte viene de la carpeta test oficial?
c = df[df.estado == "conservada"]
display(pd.crosstab(c.split_oficial, c.particion_propuesta, margins=True))
of_test = df[df.split_oficial == "test"]
print("Imágenes de la carpeta test oficial (de las 12 clases):", int(of_test.clase.isin(CLASES_EVAL_PD).sum()), "| excluidas:",
      of_test.estado.value_counts().to_dict())
# Casos del 'rango <=6' que a ojo no son duplicados (lectura de la sesión 4): efecto de aplicar la regla mecánica
for a, b in [("2643371-version", "LateBlight03.jpg")]:
    f = df[df.nombre_original.str.startswith(a) | df.nombre_original.str.startswith(b)][["ruta", "clase", "estado", "grupo_hash"]]
    print("\nPar que a ojo NO parece duplicado (sesión 4); la regla mecánica lo trata como contradictorio:"); display(f)
f = df[df.nombre_original.str.startswith("LateBlight-Leaflet-Large-Spo") | df.nombre_original.str.startswith("Tomato-late-blight-leaf-Marg")][["ruta", "clase", "estado", "copia_conservada"]]
print("Par 'dudoso' de la misma clase (se conserva una copia):"); display(f)
print(f"Tiempo total: {time.time()-T0:.0f} s | RSS: {mem_mb()} MB")

particion_propuesta,dev,test,All
split_oficial,,,
test,72,24,96
train,635,279,914
All,707,303,1010


Imágenes de la carpeta test oficial (de las 12 clases): 102 | excluidas: {'conservada': 96, 'excluida_contradictoria': 6}

Par que a ojo NO parece duplicado (sesión 4); la regla mecánica lo trata como contradictorio:


,ruta,clase,estado,grupo_hash
691,raw/plantdoc/train/Potato leaf early blight/2643371-version%3D1.0%...,Potato leaf early blight,excluida_contradictoria,H0019
844,raw/plantdoc/train/Potato leaf late blight/LateBlight03.jpg,Potato leaf late blight,excluida_contradictoria,H0019


Par 'dudoso' de la misma clase (se conserva una copia):


,ruta,clase,estado,copia_conservada
426,raw/plantdoc/train/Tomato leaf late blight/LateBlight-Leaflet-Larg...,Tomato leaf late blight,conservada,raw/plantdoc/train/Tomato leaf late blight/LateBlight-Leaflet-Larg...
457,raw/plantdoc/train/Tomato leaf late blight/Tomato-late-blight-leaf...,Tomato leaf late blight,excluida_duplicada,raw/plantdoc/train/Tomato leaf late blight/LateBlight-Leaflet-Larg...


Tiempo total: 3 s | RSS: 161 MB
